# Fine-tune LLM using LoRA and QLoRA

In [ ]:
!git clone https://github.com/shosakaue0808/LoRA-QLoRA-studies.git

In [ ]:
!pip install transformers peft bitsandbytes python-dotenv datasets

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import prepare_model_for_kbit_training
from huggingface_hub import login
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from dotenv import load_dotenv

import os
# my files
from src.lora_layer import attach_Lora_to_Linear
from src.data_prep import GSMDataset




## login Hugging face

In [2]:
load_dotenv()
token = os.getenv("HF_TOKEN")
login(token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# Set device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load Model

# LoRA base

In [3]:
model_id = "meta-llama/Llama-3.2-1B"

In [4]:


tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

c:\Users\bansh\miniconda3\envs\PEFT\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bansh\.cache\huggingface\hub\models--meta-llama--Llama-3.2-1B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

# QLoRA base

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
base_model_qlora = LlamaForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
base_model_qlora.model_name = model_id
base_tokenizer_qlora = AutoTokenizer.from_pretrained(model_id)
base_tokenizer_qlora.pad_token = base_tokenizer_qlora.eos_token
base_model_qlora.config.pad_token_id = base_tokenizer_qlora.pad_token_id

In [ ]:
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

In [ ]:
#Set all weights freeze
for param in base_model.parameters():
    param.requires_grad = False

# Attach LoRA layer to all linear layers in base model

In [ ]:
attach_Lora_to_Linear(base_model_lora, r=4, alpha=16)

In [ ]:
# make sure only A, B parameters are trainable
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

In [ ]:
# ratio of trainable / total
total_trainable_params = sum(p.numel() for p in base_model_lora.parameters() if p.requires_grad)
total_prams= sum(p.numel() for p in base_model_lora.parameters())
print(total_trainable_params/total_prams)


# Load dataset

In [5]:
ds = load_dataset("openai/gsm8k", "main")

In [6]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})


In [6]:
train_ds = ds["train"]
test_ds = ds["test"]

In [10]:
print(train_ds)

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})


In [16]:
train_dataset = GSMDataset(train_ds, tokenizer)
test_dataset = GSMDataset(test_ds, tokenizer)

Max tokens: 460
Max tokens: 416


In [14]:
print(train_dataset)

In [19]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Fine-tune process

In [ ]:
@torch.no_grad()
def evaluate(model, device, val_loader):
    model.eval()
    total_loss = 0.0
    total_batches = 0

    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # skip batches with no valid labels
        if (labels != -100).sum() == 0:
           continue

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss

        # skip NaN / Inf
        if not torch.isfinite(loss):
            continue

        total_loss += loss.item()
        total_batches += 1

    model.train()
    return total_loss / max(total_batches, 1)

In [ ]:
#training
def train_lora(model, optimizer, train_loader, val_loader, epochs, device, store_bit):
    train_losses = []
    val_losses = []
    steps = []

    global_step = 0
    check_every = 200   # save/check every 200 steps
    best_val_loss = float('inf')
    model.train()
    for epoch in range(epochs):
        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            global_step += 1

            if global_step % check_every == 0:
                train_loss = loss.item()
                val_loss = evaluate(model, device, val_loader)

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                steps.append(global_step)

                print(
                    f"epoch {epoch+1} | step {global_step} | "
                    f"train_loss {train_loss:.4f} | val_loss {val_loss:.4f}"
                )

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    save_checkpoint(
                        path=f"{checkpoint_dir}/llama_2_7B_Manually_{model.rank}_{store_bit}_checkpoint_step_{global_step}.pt",
                        model=model,
                        optimizer=optimizer,
                        epoch=epoch,
                        global_step=global_step,
                        train_loss=train_loss,
                        val_loss=val_loss,
                    )
    return train_losses, val_losses, steps